# **Handling Date and Time Data**

## Topic Roadmap

- **1. Imports and Dataset Loading**
  - 1.1 Load CSV Files
  - 1.2 Convert to Datetime Datatype
- **2. Date Feature Extraction**
  - 2.1 Extract Year, Month, and Day
  - 2.2 Extract Day of Week and Weekend Indicator
  - 2.3 Extract Week, Quarter, and Semester
  - 2.4 Calculate Elapsed Time Between Dates
- **3. Time Feature Extraction**
  - 3.1 Extract Hour, Minute, and Second
  - 3.2 Extract Pure Time Object
  - 3.3 Calculate Time Differences in Intervals
- **4. Key Revision Notes**

## **1. Imports and Dataset Loading**

### **Load CSV Files and Convert Types**

Raw date and time fields are loaded as string (`object`) data types by default[cite: 2]. Converting these columns to Pandas `datetime64` enables vectorized temporal extraction and elapsed time calculations[cite: 2].

In [2]:
import datetime
import numpy as np
import pandas as pd

# Load datasets
date_df = pd.read_csv('docs/Lecture-021-orders.csv')
time_df = pd.read_csv('docs/Lecture-021-messages.csv')

# Convert date columns to datetime datatype
date_df['date'] = pd.to_datetime(date_df['date'])
time_df['date'] = pd.to_datetime(time_df['date'])

date_df.info()
time_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        1000 non-null   datetime64[us]
 1   product_id  1000 non-null   int64         
 2   city_id     1000 non-null   int64         
 3   orders      1000 non-null   int64         
dtypes: datetime64[us](1), int64(3)
memory usage: 31.4 KB
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    1000 non-null   datetime64[us]
 1   msg     1000 non-null   str           
dtypes: datetime64[us](1), str(1)
memory usage: 100.1 KB


## **2. Date Feature Extraction**

### **Extracting Calendar Attributes and Time Passed**

The `.dt` accessor extracts components such as year, month, day, day of week, week number, quarter, and semester from a datetime series[cite: 2]. Subtracting dates from a reference timestamp yields time elapsed[cite: 2].

In [3]:
# Extract calendar units
date_df['date_year'] = date_df['date'].dt.year
date_df['date_month_no'] = date_df['date'].dt.month
date_df['date_month_name'] = date_df['date'].dt.month_name()
date_df['date_day'] = date_df['date'].dt.day

# Extract day of week features
date_df['date_dow'] = date_df['date'].dt.dayofweek
date_df['date_dow_name'] = date_df['date'].dt.day_name()
date_df['date_is_weekend'] = np.where(date_df['date_dow_name'].isin(['Sunday', 'Saturday']), 1, 0)

# Extract extended intervals
date_df['date_week'] = date_df['date'].dt.isocalendar().week
date_df['quarter'] = date_df['date'].dt.quarter
date_df['semester'] = np.where(date_df['quarter'].isin([1, 2]), 1, 2)

# Calculate elapsed time
today = datetime.datetime.today()
date_df['elapsed_days'] = (today - date_df['date']).dt.days
date_df['elapsed_months'] = np.round((today - date_df['date']) / pd.Timedelta(days=30.4375), 0)

date_df.head()

,date,product_id,city_id,orders,date_year,date_month_no,date_month_name,date_day,date_dow,date_dow_name,date_is_weekend,date_week,quarter,semester,elapsed_days,elapsed_months
0,2019-12-10,5628,25,3,2019,12,December,10,1,Tuesday,0,50,4,2,2423,80.0
1,2018-08-15,3646,14,157,2018,8,August,15,2,Wednesday,0,33,3,2,2905,95.0
2,2018-10-23,1859,25,1,2018,10,October,23,1,Tuesday,0,43,4,2,2836,93.0
3,2019-08-17,7292,25,1,2019,8,August,17,5,Saturday,1,33,3,2,2538,83.0
4,2019-01-06,4344,25,3,2019,1,January,6,6,Sunday,1,1,1,1,2761,91.0


## **3. Time Feature Extraction**

### **Extracting Time Units and High-Granularity Differences**

High-resolution timestamps allow extraction of hours, minutes, seconds, and standard `datetime.time` objects[cite: 2]. Duration calculation converts timedeltas into explicit intervals like seconds, minutes, and hours[cite: 2].

In [4]:
# Extract time components
time_df['hour'] = time_df['date'].dt.hour
time_df['min'] = time_df['date'].dt.minute
time_df['sec'] = time_df['date'].dt.second
time_df['time'] = time_df['date'].dt.time

# Calculate time differences in specific units
time_df['diff_seconds'] = (today - time_df['date']) / pd.Timedelta(seconds=1)
time_df['diff_minutes'] = (today - time_df['date']) / pd.Timedelta(minutes=1)
time_df['diff_hours'] = (today - time_df['date']) / pd.Timedelta(hours=1)

time_df.head()

,date,msg,hour,min,sec,time,diff_seconds,diff_minutes,diff_hours
0,2013-12-15 00:50:00,ищу на сегодня мужика 37,0,50,0,00:50:00,3.982614e+08,6.637691e+06,110628.176034
1,2014-04-29 23:40:00,ПАРЕНЬ БИ ИЩЕТ ДРУГА СЕЙЧАС!! СМС ММС 0955532826,23,40,0,23:40:00,3.865152e+08,6.441921e+06,107365.342701
2,2012-12-30 00:21:00,Днепр.м 43 позн.с д/ж *.о 067.16.34.576,0,21,0,00:21:00,4.285032e+08,7.141720e+06,119028.659368
3,2014-11-28 00:31:00,КИЕВ ИЩУ Д/Ж ДО 45 МНЕ СЕЙЧАС СКУЧНО 093 629 9...,0,31,0,00:31:00,3.681954e+08,6.136590e+06,102276.492701
4,2013-10-26 23:11:00,Зая я тебя никогда не обижу люблю тебя!) Даше,23,11,0,23:11:00,4.025010e+08,6.708350e+06,111805.826034


## **Key Revision Notes**

- **Type Casting**: Convert string columns to datetime using `pd.to_datetime()` before extracting attributes[cite: 2].
- **Vectorized Extraction**: Use `.dt` to access properties (`dt.year`, `dt.month`, `dt.day`, `dt.hour`, `dt.minute`, `dt.second`) directly on Pandas series[cite: 2].
- **Name Extraction**: Use `dt.month_name()` and `dt.day_name()` for string representations of months and days[cite: 2].
- **Day of Week**: `dt.dayofweek` returns integer representations (0 = Monday, 6 = Sunday)[cite: 2].
- **Week Extraction**: Use `dt.isocalendar().week` to get ISO-compliant week numbers[cite: 2].
- **Timedelta Operations**: Subtracting datetimes produces timedelta objects[cite: 2]. Access duration with `.dt.days` or divide by `pd.Timedelta` to convert to units like seconds, minutes, or hours[cite: 2].